# NB06 — PGLS with genome-derived Levins B

**Goal:** Replace the MicrobeAtlas 16S-derived niche breadth (P1 response) with genome-derived Levins' B
computed from the distribution of SPIRE MAGs across sampling-site biome categories.

**Question addressed:** Does the P1 PGLS result (metal gene density × niche breadth, β=−0.021, p<0.05)
change when niche breadth is derived from genome sampling locations rather than 16S OTU detections?

**Design (v2):**
- Response: Levins' B per genus from SPIRE MAG biome-category distributions (all-env set, 36 ENVO classes)
- Predictor: genus-median `ko_per_mb_primary` from SPIRE soil MAGs (same as NB04)
- Tree: Full GTDB R214 genus tree (13,675 genera; built from bac120 + taxonomy outside notebook)
- Filter: >=5 MAGs with non-empty biome annotation per genus
- No ke_pangenome data used

**Comparison to prior analyses:**

| Analysis | Response | Predictor | Tree | n genera |
|---|---|---|---|---|
| P1 | MicrobeAtlas 16S Levins B | Pangenome | P1 pruned (2,283) | ~1,574 |
| NB04 | MicrobeAtlas 16S Levins B | SPIRE MAGs | P1 pruned (2,283) | 254 |
| NB06 v1 | SPIRE genome biome Levins B | SPIRE MAGs | P1 pruned (2,283) | 94 |
| **NB06 v2 (this)** | **SPIRE genome biome Levins B** | **SPIRE MAGs** | **GTDB full (13,675)** | **~370** |

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import dendropy
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H
apply_style()

_REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(_REPO_ROOT / 'comprehensive_metal_ecology' / 'scripts'))
from pgls_utils import run_pgls, pgls_results_table, fdr_correct

DATA_DIR = Path.cwd().parent / 'data'
FIGS     = Path.cwd().parent / 'figures'

TREE_PATH = DATA_DIR / 'gtdb_bac_genus_full.tree'
MIN_MAGS_PER_GENUS = 5

print('NB06 v2 executing.')

NB06 v2 executing.


In [2]:
fm       = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')
meta_all = pd.read_csv(DATA_DIR / 'all_env_mag_metadata_cache.csv')
samp     = pd.read_parquet(DATA_DIR / 'spire_sample_metadata.parquet')

meta_biome = meta_all.merge(samp[['sample_id', 'microntology']], on='sample_id', how='left')
biome_df = meta_biome[
    meta_biome['microntology'].notna() & (meta_biome['microntology'] != '')
].copy()

print(f'Feature matrix (KO source): {len(fm):,} MAGs, {fm["genus"].nunique():,} genera')
print(f'All-env MAGs with biome: {len(biome_df):,} ({len(biome_df)/len(meta_all)*100:.1f}%)')
print(f'Unique biome categories: {biome_df["microntology"].nunique()}')

Feature matrix (KO source): 15,957 MAGs, 1,934 genera
All-env MAGs with biome: 19,198 (13.0%)
Unique biome categories: 37


In [3]:
def levins_b(series):
    counts = series.value_counts()
    n = counts.sum()
    if n < 2:
        return np.nan
    p = counts / n
    return 1.0 / (p ** 2).sum()

genus_levins = (
    biome_df
    .groupby('genus')['microntology']
    .agg(levins_b_genome=levins_b, n_mags_biome='count')
    .reset_index()
)
genus_levins = genus_levins[genus_levins['n_mags_biome'] >= MIN_MAGS_PER_GENUS].copy()

print(f'Genera with >={MIN_MAGS_PER_GENUS} biome MAGs: {len(genus_levins)}')
print(f'Levins B range: {genus_levins["levins_b_genome"].min():.3f} - {genus_levins["levins_b_genome"].max():.3f}')
print(f'Median: {genus_levins["levins_b_genome"].median():.3f}')

Genera with >=5 biome MAGs: 643
Levins B range: 1.000 - 10.974
Median: 2.270


In [4]:
genus_ko = (
    fm
    .groupby('genus')
    .agg(ko_per_mb_primary=('ko_per_mb_primary', 'median'), n_mags_total=('mag_id', 'count'))
    .reset_index()
)
print(f'Genera with KO density: {len(genus_ko):,}')

Genera with KO density: 1,934


In [5]:
gtdb_tree = dendropy.Tree.get(path=str(TREE_PATH), schema='newick', preserve_underscores=True)
tree_genera = {n.taxon.label for n in gtdb_tree.leaf_node_iter() if n.taxon}
print(f'GTDB genus tree: {len(tree_genera):,} genera')

genus_levins['genus'] = genus_levins['genus'].str.lower()
genus_ko['genus']     = genus_ko['genus'].str.lower()

feat = genus_levins.merge(genus_ko, on='genus', how='inner')
print(f'After Levins B + KO join: {len(feat)} genera')

feat = feat[feat['genus'].isin(tree_genera)].copy()
print(f'After GTDB tree filter: {len(feat)} genera')

feat['levins_b_genome_z']   = stats.zscore(feat['levins_b_genome'],   nan_policy='omit')
feat['ko_per_mb_primary_z'] = stats.zscore(feat['ko_per_mb_primary'], nan_policy='omit')
feat = feat.dropna(subset=['levins_b_genome_z', 'ko_per_mb_primary_z'])
print(f'Final n (complete cases): {len(feat)}')

GTDB genus tree: 13,675 genera
After Levins B + KO join: 640 genera
After GTDB tree filter: 328 genera
Final n (complete cases): 328


In [6]:
pgls_out = run_pgls(
    df=feat,
    tree_path=str(TREE_PATH),
    response='levins_b_genome_z',
    predictors=['ko_per_mb_primary_z'],
    taxon_col='genus',
)

results_df = pgls_results_table([pgls_out])
results_df['p_value_fdr'] = fdr_correct(results_df['p_value'].dropna())
results_df.to_csv(DATA_DIR / 'nb06_genome_levins_b_pgls.csv', index=False)

print(results_df.to_string(index=False))

row = results_df[results_df['predictor'] == 'ko_per_mb_primary_z'].iloc[0]
ko_beta, ko_se, ko_p, ko_n, ko_lam = (
    float(row['beta']), float(row['SE']), float(row['p_value']),
    int(row['n']), float(results_df['lambda_est'].iloc[0])
)

P1_BETA = -0.021; NB04_BETA = -0.011; NB06V1_BETA = 0.135
print(f'\nNB06 v2: beta={ko_beta:.4f}, SE={ko_se:.4f}, p={ko_p:.4f}, n={ko_n}, lambda={ko_lam:.4f}')
print(f'Sign consistent with P1  (beta={P1_BETA}):    {"YES" if np.sign(ko_beta)==np.sign(P1_BETA) else "NO"}')
print(f'Sign consistent with NB04 (beta={NB04_BETA}): {"YES" if np.sign(ko_beta)==np.sign(NB04_BETA) else "NO"}')

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


label          response           predictor   n  lambda_est      beta       SE    t_stat  p_value       r2  delta_aic_vs_null  p_value_fdr
      levins_b_genome_z ko_per_mb_primary_z 328      0.2042 -0.055576 0.066977 -0.829776 0.407273 0.000482               1.31     0.407273

NB06 v2: beta=-0.0556, SE=0.0670, p=0.4073, n=328, lambda=0.2042
Sign consistent with P1  (beta=-0.021):    YES
Sign consistent with NB04 (beta=-0.011): YES


In [7]:
nb04_results = pd.read_csv(DATA_DIR / 'pgls_validation_results.csv')
nb04_ko = nb04_results[nb04_results['predictor'] == 'ko_per_mb_primary_z'].iloc[0]

labels = ['P1\n(MicrobeAtlas 16S\nniche breadth)',
          'NB04\n(SPIRE KO +\nMicrobeAtlas breadth)',
          'NB06 v1\n(SPIRE biome,\nP1 tree, n=94)',
          'NB06 v2\n(SPIRE biome,\nGTDB tree)']
betas  = [P1_BETA, float(nb04_ko['beta']), NB06V1_BETA, ko_beta]
ses    = [None,    float(nb04_ko['SE']),   0.098,       ko_se]
pvals  = [2e-8,   float(nb04_ko['p_value']), 0.172,    ko_p]
ns     = [1574,   int(nb04_ko['n']),        94,        ko_n]
colors = [PALETTE[0], PALETTE[1], PALETTE[3], PALETTE[2]]

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H * 1.2))

for i, (beta, se, pval, n, col) in enumerate(zip(betas, ses, pvals, ns, colors)):
    kw = dict(color=col, edgecolor='k', linewidth=0.5, height=0.55)
    if se is not None:
        ax.barh(i, beta, xerr=1.96 * se, error_kw={'elinewidth': 1.0, 'capsize': 3}, **kw)
    else:
        ax.barh(i, beta, **kw)
    sig = '***' if pval < 0.001 else ('*' if pval < 0.05 else 'ns')
    offset = 0.003 if beta >= 0 else -0.003
    ax.text(beta + offset, i, f'p={pval:.3f} {sig}  n={n}',
            va='center', ha='left' if beta >= 0 else 'right', fontsize=8, color='#808080')

ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('PGLS beta (metal gene density -> niche breadth)', fontsize=9)
ax.set_ylabel('')
ax.set_title('Metal gene density x niche breadth: sensitivity analysis', fontsize=10)
fig.suptitle('', y=1.02)

save(fig, FIGS / 'nb06_pgls_comparison')
print('Saved figures/nb06_pgls_comparison.pdf')

Saved figures/nb06_pgls_comparison.pdf


In [8]:

# Aggregate all KO subcategories at genus level (using the same feature matrix)
KO_CATS = ['ko_per_mb_cofactor', 'ko_per_mb_resistance', 'ko_per_mb_transport',
           'ko_per_mb_sensing', 'ko_per_mb_metabolism']

genus_ko_cats = (
    fm.groupby('genus')
    .agg({col: 'median' for col in KO_CATS})
    .reset_index()
)
genus_ko_cats['genus'] = genus_ko_cats['genus'].str.lower()
print(f'Genus-level KO categories aggregated: {len(genus_ko_cats)} genera')
print(genus_ko_cats[KO_CATS].describe().round(3).to_string())


Genus-level KO categories aggregated: 1934 genera
       ko_per_mb_cofactor  ko_per_mb_resistance  ko_per_mb_transport  ko_per_mb_sensing  ko_per_mb_metabolism
count            1934.000              1934.000             1934.000           1934.000              1934.000
mean                1.685                 8.423               17.589              3.833                 2.946
std                 1.518                 7.273               14.998              3.315                 2.622
min                 0.000                 1.081                1.250              0.000                 0.000
25%                 0.854                 4.310                8.871              1.972                 1.514
50%                 1.232                 6.028               12.969              2.767                 2.036
75%                 1.835                 8.756               19.441              4.102                 2.977
max                12.553                54.854              126.433  

In [9]:

# Per-category PGLS: genome-derived Levins B ~ each KO subcategory
# P1 reference values from data/03_category_pgls_results.csv (MicrobeAtlas 16S response, n~928-1073)
P1_CATS = {
    'ko_per_mb_cofactor':   {'beta': -0.032736, 'p': 1.033e-9,  'n': 928},
    'ko_per_mb_resistance': {'beta': +0.002523, 'p': 6.556e-1,  'n': 1073},
    'ko_per_mb_transport':  {'beta': -0.021781, 'p': 1.102e-5,  'n': 1073},
    'ko_per_mb_sensing':    {'beta': -0.018449, 'p': 7.251e-4,  'n': 1069},
    'ko_per_mb_metabolism': {'beta': -0.020903, 'p': 7.471e-5,  'n': 1056},
}

cat_results = []
for cat in KO_CATS:
    feat_cat = (
        feat[['genus', 'levins_b_genome_z']]
        .merge(genus_ko_cats[['genus', cat]], on='genus', how='inner')
        .dropna()
        .copy()
    )
    z_col = f'{cat}_z'
    feat_cat[z_col] = stats.zscore(feat_cat[cat], nan_policy='omit')
    feat_cat = feat_cat.dropna(subset=[z_col])

    res = run_pgls(
        df=feat_cat, tree_path=str(TREE_PATH),
        response='levins_b_genome_z',
        predictors=[z_col],
        taxon_col='genus',
    )
    row = pgls_results_table([res]).iloc[0]
    cat_results.append({
        'category':     cat.replace('ko_per_mb_', ''),
        'predictor_col': cat,
        'n':            int(row['n']),
        'beta':         float(row['beta']),
        'se':           float(row['SE']),
        'p':            float(row['p_value']),
        'lambda_est':   float(row['lambda_est']),
        'p1_beta':      P1_CATS[cat]['beta'],
        'p1_p':         P1_CATS[cat]['p'],
        'p1_n':         P1_CATS[cat]['n'],
    })
    last = cat_results[-1]
    print(f"{cat:30s}: n={last['n']:3d}, beta={last['beta']:+.4f}, "
          f"p={last['p']:.4f}, lambda={last['lambda_est']:.3f}  "
          f"[P1: beta={P1_CATS[cat]['beta']:+.4f}, p={P1_CATS[cat]['p']:.2e}]")

cat_df = pd.DataFrame(cat_results)
cat_df.to_csv(DATA_DIR / 'nb06_category_pgls_results.csv', index=False)
print('\nSaved nb06_category_pgls_results.csv')
print(cat_df[['category', 'n', 'beta', 'se', 'p', 'lambda_est', 'p1_beta', 'p1_p']].to_string(index=False))


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


ko_per_mb_cofactor            : n=328, beta=-0.0723, p=0.2461, lambda=0.200  [P1: beta=-0.0327, p=1.03e-09]


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


ko_per_mb_resistance          : n=328, beta=-0.0501, p=0.4629, lambda=0.204  [P1: beta=+0.0025, p=6.56e-01]


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


ko_per_mb_transport           : n=328, beta=-0.0659, p=0.2929, lambda=0.198  [P1: beta=-0.0218, p=1.10e-05]


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


ko_per_mb_sensing             : n=328, beta=-0.0562, p=0.3924, lambda=0.204  [P1: beta=-0.0184, p=7.25e-04]


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


ko_per_mb_metabolism          : n=328, beta=-0.0396, p=0.5769, lambda=0.204  [P1: beta=-0.0209, p=7.47e-05]

Saved nb06_category_pgls_results.csv
  category   n      beta       se        p  lambda_est   p1_beta         p1_p
  cofactor 328 -0.072315 0.062229 0.246051      0.1999 -0.032736 1.033000e-09
resistance 328 -0.050067 0.068127 0.462922      0.2045  0.002523 6.556000e-01
 transport 328 -0.065947 0.062594 0.292859      0.1979 -0.021781 1.102000e-05
   sensing 328 -0.056201 0.065623 0.392387      0.2038 -0.018449 7.251000e-04
metabolism 328 -0.039608 0.070921 0.576898      0.2038 -0.020903 7.471000e-05


In [10]:

# Forest plot: per-category PGLS — NB06 v2 (genome Levins B) vs P1 (MicrobeAtlas Levins B)
CAT_LABELS = {
    'cofactor':   'Cofactor\n(homeostasis)',
    'resistance': 'Resistance\n(inducible)',
    'transport':  'Transport',
    'sensing':    'Sensing',
    'metabolism': 'Metabolism',
}

# Order: cofactor first (expected signal), resistance last (expected null)
cat_order = ['cofactor', 'transport', 'sensing', 'metabolism', 'resistance']
cat_df_ord = cat_df.set_index('category').loc[cat_order].reset_index()
n_cats = len(cat_df_ord)

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H * 1.2),
                         sharey=True, gridspec_kw={'wspace': 0.04})

for ax, (beta_col, se_col, p_col, n_col, title, color) in zip(
        axes,
        [('p1_beta', None,   'p1_p', 'p1_n',  'P1 — MicrobeAtlas 16S\nniche breadth', PALETTE[1]),
         ('beta',    'se',   'p',    'n',      'NB06 v2 — SPIRE genome\nniche breadth', PALETTE[0])]):
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    for i, row in cat_df_ord.iterrows():
        y     = n_cats - 1 - i
        beta  = row[beta_col]
        se    = row[se_col] if se_col else None
        pval  = row[p_col]
        n     = row[n_col]
        kw    = dict(color=color, edgecolor='k', linewidth=0.5, height=0.55)
        if se is not None:
            ax.barh(y, beta, xerr=1.96 * se, error_kw={'elinewidth': 1.0, 'capsize': 3}, **kw)
        else:
            ax.barh(y, beta, **kw)
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        pad = max(abs(beta) * 0.08, 0.002)
        ha  = 'left' if beta >= 0 else 'right'
        ax.text(beta + (pad if beta >= 0 else -pad), y,
                f'{sig}  n={n:,}', va='center', ha=ha, fontsize=8, color='#808080')
    ax.set_yticks(range(n_cats))
    ax.set_yticklabels(
        [CAT_LABELS.get(cat_df_ord.iloc[n_cats-1-k]['category'], '') for k in range(n_cats)],
        fontsize=8
    )
    ax.set_xlabel('PGLS β (metal gene density → niche breadth)', fontsize=9)
    ax.set_title(title, fontsize=10)

fig.suptitle('Per-category PGLS: MicrobeAtlas 16S vs SPIRE genome niche breadth', y=1.04)
save(fig, FIGS / 'nb06_category_forest_plot')
print('Saved figures/nb06_category_forest_plot.pdf')


Saved figures/nb06_category_forest_plot.pdf


In [11]:
summary = {
    'notebook': 'NB06_v2',
    'n_mags_feature_matrix': int(len(fm)),
    'n_mags_all_env_with_biome': int(len(biome_df)),
    'n_biome_categories': int(biome_df['microntology'].nunique()),
    'n_genera_levins_eligible': int(len(genus_levins)),
    'n_genera_in_gtdb_tree': len(tree_genera),
    'min_mags_per_genus': MIN_MAGS_PER_GENUS,
    'n_genera_pgls': ko_n,
    'beta_nb06_v2': ko_beta, 'se_nb06_v2': ko_se, 'p_nb06_v2': ko_p, 'lambda_nb06_v2': ko_lam,
    'beta_nb06_v1': NB06V1_BETA, 'p_nb06_v1': 0.172, 'n_nb06_v1': 94,
    'beta_p1': P1_BETA, 'p_p1': 2e-8,
    'beta_nb04': float(NB04_BETA), 'p_nb04': float(nb04_ko['p_value']),
    'sign_consistent_p1': bool(np.sign(ko_beta) == np.sign(P1_BETA)),
    'sign_consistent_nb04': bool(np.sign(ko_beta) == np.sign(NB04_BETA)),
}
with open(DATA_DIR / 'nb06_build_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "notebook": "NB06_v2",
  "n_mags_feature_matrix": 15957,
  "n_mags_all_env_with_biome": 19198,
  "n_biome_categories": 37,
  "n_genera_levins_eligible": 643,
  "n_genera_in_gtdb_tree": 13675,
  "min_mags_per_genus": 5,
  "n_genera_pgls": 328,
  "beta_nb06_v2": -0.055576143154999945,
  "se_nb06_v2": 0.06697729666198717,
  "p_nb06_v2": 0.40727272654383917,
  "lambda_nb06_v2": 0.2042,
  "beta_nb06_v1": 0.135,
  "p_nb06_v1": 0.172,
  "n_nb06_v1": 94,
  "beta_p1": -0.021,
  "p_p1": 2e-08,
  "beta_nb04": -0.011,
  "p_nb04": 0.2160588488634545,
  "sign_consistent_p1": true,
  "sign_consistent_nb04": true
}
